# The two main questions we answer (the business requirement / problem)
- Environmental impact on accidents (location, weather/season, time of day)
- Manufacturer Confidence Analysis (Can they be trusted)

In [ ]:
# sources:
# https://aviation-safety.net/database/

In [ ]:
import pandas as pd
import numpy as np

## Dataframe 1

In [ ]:
url = "https://raw.githubusercontent.com/oshani-jayawardane/aircraft-accident-project/main/aviation_accidents_2000_2025_merged.csv"
df = pd.read_csv(url)

df.head()

,Event No.,Year,List Date,Detail URL,Time,Type,Owner/operator,Registration,MSN,Total airframe hrs,...,Location,Phase,Nature,Departure airport,Destination airport,Investigating agency,Confidence Rating,Narrative,Year of manufacture,Cycles
0,1,2000,1 Jan 2000,https://aviation-safety.net/wikibase/299408,13:00 LT,Cessna 550 Citation II,US Customs Service,N752CC,550-0018,12159 hours,...,"HOMESTEAD, Florida - United States of America",Unknown,Ferry/positioning,"MIAMI , FL (KTMB)",(KHST),NTSB,Accident investigation report completed and in...,The pilot-in-command (PIC) stated he was in cr...,NaN,NaN
1,2,2000,3 Jan 2000,https://aviation-safety.net/wikibase/323589,NaN,Beechcraft 200 Super King Air,Kalahari Air Services & Charter,A2-AEZ,BB-421,NaN,...,unknown location - Unknown country,Unknown,Unknown,NaN,NaN,NaN,Little or no information is available,Damaged beyond repair.,1978.0,NaN
2,3,2000,4 Jan 2000,https://aviation-safety.net/wikibase/299403,17:25 LT,Beechcraft B200 Super King Air,Private,N895TT,BB-1239,3238 hours,...,"JACKSON, Wyoming - United States of America",Unknown,Private,"GRAND ISLAND , NE (KGRI)",(KJAC),NTSB,Accident investigation report completed and in...,While performing the ILS runway 18 approach to...,1986.0,NaN
3,4,2000,5 Jan 2000,https://aviation-safety.net/wikibase/323588,13:25,Embraer EMB-110P1A Bandeirante,Skypower Express Airways,5N-AXL,110455,NaN,...,Abuja International Airport (ABV) - Nigeria,Approach,Passenger - Scheduled,Lagos-Murtala Muhammed International Airport (...,Abuja International Airport (ABV/DNAA),NaN,Information verified through data from acciden...,The Bandeirante aircraft was coming in to land...,1984.0,NaN
4,5,2000,7 Jan 2000,https://aviation-safety.net/wikibase/323587,NaN,Antonov An-26,Unknown,D2-FBR,7206,NaN,...,- Angola,Unknown,Cargo,Luanda-4 de Fevereiro Airport (LAD/FNLU),Cafunfo Airport (CFF/FNCF),NaN,NaN,Disappeared near the border of the Angolan pro...,1978.0,NaN


In [ ]:
df.shape

(6848, 25)

In [ ]:
# convert to datetime -- here safe to convert casue we already have a seperate year column

df['List Date'] = pd.to_datetime(df['List Date'], format='%d %b %Y', errors='coerce')
df.head(2)

,Event No.,Year,List Date,Detail URL,Time,Type,Owner/operator,Registration,MSN,Total airframe hrs,...,Location,Phase,Nature,Departure airport,Destination airport,Investigating agency,Confidence Rating,Narrative,Year of manufacture,Cycles
0,1,2000,2000-01-01,https://aviation-safety.net/wikibase/299408,13:00 LT,Cessna 550 Citation II,US Customs Service,N752CC,550-0018,12159 hours,...,"HOMESTEAD, Florida - United States of America",Unknown,Ferry/positioning,"MIAMI , FL (KTMB)",(KHST),NTSB,Accident investigation report completed and in...,The pilot-in-command (PIC) stated he was in cr...,NaN,NaN
1,2,2000,2000-01-03,https://aviation-safety.net/wikibase/323589,NaN,Beechcraft 200 Super King Air,Kalahari Air Services & Charter,A2-AEZ,BB-421,NaN,...,unknown location - Unknown country,Unknown,Unknown,NaN,NaN,NaN,Little or no information is available,Damaged beyond repair.,1978.0,NaN


In [ ]:
# there are this many date unknown entries, but the year column has the year

df['List Date'].isna().sum()

np.int64(61)

In [ ]:
df.columns

In [ ]:
# drop unwanted columns
df = df.drop(columns=['Detail URL', 'MSN', 'Investigating agency', 'Cycles', 'Other fatalities'], errors='ignore')

# rename columns
df = df.rename(columns={
    'Event No.'           : 'Event_No',
    'List Date'           : 'Incident_Date',
    'Type'                : 'Aircaft_Model',
    'Registration'        : 'Aircaft_Registration',
    'Owner/operator'      : 'Aircaft_Operator',
    'Total airframe hrs'  : 'Total_Flight_Hours',
    'Engine model'        : 'Engine_Model',
    'Aircraft damage'     : 'Aircaft_Damage_Type',
    'Category'            : 'Incident_Category',
    'Location'            : 'Incident_Location',
    'Phase'               : 'Aircraft_Phase',
    'Nature'              : 'Aircaft_Nature',
    'Departure airport'   : 'Departure_Airport',
    'Destination airport' : 'Destination_Airport',
    'Confidence Rating'   : 'Confidence_Rating',
    'Year of manufacture' : 'Aircraft_Manufacture_Year'
})

desired_order = [
    'Event_No', 'Incident_Date', 'Year', 'Time',
    'Incident_Category', 'Aircaft_Damage_Type', 'Aircraft_Phase', 'Incident_Location', 'Fatalities',
    'Aircaft_Nature', 'Aircaft_Registration', 'Aircaft_Model', 'Aircaft_Operator', 'Engine_Model',
    'Departure_Airport', 'Destination_Airport', 'Aircraft_Manufacture_Year', 'Total_Flight_Hours',
    'Confidence_Rating', 'Narrative'
]

# keep the columns in your desired order and then append the rest (so nothing gets lost)
df = df[desired_order + [col for col in df.columns if col not in desired_order]]

# drop duplicates
df = df.drop_duplicates()

df.head(3)

## Dataframe 2

In [ ]:
url2 = "https://raw.githubusercontent.com/oshani-jayawardane/aircraft-accident-project/main/Aircraft_Incident_Dataset.csv"
df2 = pd.read_csv(url2)

df2.head()

,Incident_Date,Aircaft_Model,Aircaft_Registration,Aircaft_Operator,Aircaft_Nature,Incident_Category,Incident_Cause(es),Incident_Location,Aircaft_Damage_Type,Date,...,Onboard_Crew,Onboard_Passengers,Onboard_Total,Fatalities,Aircaft_First_Flight,Aircraft_Phase,Departure_Airport,Destination_Airport,Ground_Casualties,Collision_Casualties
0,03-JAN-2022,British Aerospace 4121 Jetstream 41,ZS-NRJ,SA Airlink,Domestic Non Scheduled Passenger,Accident | repairable-damage,"Airplane - Engines, Airplane - Engines - Prop/...",near Venetia Mine...,Substantial,Monday 3 January 2022,...,Fatalities: 0 / Occupants: 3,Fatalities: 0 / Occupants: 4,Fatalities: 0 / Occupants: 7,0,1995-05-19 (26 years 8 months),Landing (LDG),Johannesburg-O.R. Tambo International Airport ...,"Venetia Mine Airport (FAVM) , South Africa",NaN,NaN
1,04-JAN-2022,British Aerospace 3101 Jetstream 31,HR-AYY,LANHSA,Domestic Scheduled Passenger,Accident | repairable-damage,"Airplane - Undercarriage, Airplane - Undercarr...",Roatán-Juan ...,Substantial,Tuesday 4 January 2022,...,Fatalities: 0 / Occupants:,Fatalities: 0 / Occupants:,Fatalities: 0 / Occupants: 19,0,1985,Landing (LDG),La Ceiba-Goloson International Airport (LCE/MH...,Roatán-Juan Manuel Gálvez International Airpor...,NaN,NaN
2,05-JAN-2022,Boeing 737-4H6,EP-CAP,Caspian Airlines,Domestic Scheduled Passenger,Accident | repairable-damage,"Airplane - Undercarriage, Airplane - Undercarr...",Isfahan-Shah...,Substantial,Wednesday 5 January 2022,...,Fatalities: 0 / Occupants:,Fatalities: 0 / Occupants:,Fatalities: 0 / Occupants: 116,0,1992-09-18 (29 years 4 months),Landing (LDG),"Mashhad Airport (MHD/OIMM) , Iran","Isfahan-Shahid Beheshti Airport (IFN/OIFM) , Iran",NaN,NaN
3,08-JAN-2022,Tupolev Tu-204-100C,RA-64032,"Cainiao, opb Aviastar-TU",Cargo,Accident | hull-loss,"Cargo - Fire/smoke, Result - Damaged on the gr...",Hangzhou-Xia...,Destroyed,Saturday 8 January 2022,...,Fatalities: 0 / Occupants: 8,Fatalities: 0 / Occupants: 0,Fatalities: 0 / Occupants: 8,0,2002-07-18 (19 years 6 months),Standing (STD),Hangzhou-Xiaoshan International Airport (HGH/Z...,"Novosibirsk-Tolmachevo Airport (OVB/UNNT) , Ru...",NaN,NaN
4,12-JAN-2022,Beechcraft 200 Super King Air,NaN,private,Illegal Flight,"Criminal occurrence (sabotage, shoot down) | h...",Result - Damaged on the ground,"Machakilha, ...",Damaged beyond repair,Wednesday 12 January 2022,...,Fatalities: 0 / Occupants: 0,Fatalities: 0 / Occupants: 0,Fatalities: 0 / Occupants: 0,0,NaN,Standing (STD),?,?,NaN,NaN


In [ ]:
df.shape

(6848, 25)

In [ ]:
df2.columns

In [ ]:
# drop unnecessary columns from df2

df2 = df2.drop(columns=['Onboard_Crew', 'Onboard_Passengers', 'Fatalities', 'Date', 'Arit'], errors='ignore')
df2.head(2)

In [ ]:
# extract year - since Incident_Date has some missing dates

df2['Year'] = df2['Incident_Date'].astype(str).str.extract(r'(\d{4})').astype('Int64')
df2['Year'].isna().sum()

In [ ]:
# now convert incident date to date time format
df2['Incident_Date'] = pd.to_datetime(df2['Incident_Date'], format='%d-%b-%Y', errors='coerce')

In [ ]:
# 1) Rename columns in df2
df2 = df2.rename(columns={
    'Onboard_Total'        : 'Fatalities',
    'Aircaft_Engines'      : 'Engine_Model',
    'Aircaft_First_Flight' : 'Aircraft_Manufacture_Year'
})

# 2) Reorder columns in the desired order
desired_order_df2 = [
    'Incident_Date', 'Year', 'Time',
    'Incident_Category', 'Aircaft_Damage_Type', 'Aircraft_Phase', 'Incident_Location',
    'Fatalities', 'Ground_Casualties', 'Collision_Casualties', 'Incident_Cause(es)',
    'Aircaft_Nature', 'Aircaft_Registration', 'Aircaft_Model', 'Aircaft_Operator', 'Engine_Model',
    'Departure_Airport', 'Destination_Airport', 'Aircraft_Manufacture_Year'
]

df2 = df2[desired_order_df2 + [col for col in df2.columns if col not in desired_order_df2]]

# drop duplicates
df2 = df2.drop_duplicates()

df2.head(2)

In [ ]:
df2.columns

In [ ]:
cols = ['Departure_Airport', 'Destination_Airport']

for c in cols:
    df2[c] = df2[c].replace(r'^\?+$', np.nan, regex=True)

# Data Cleaning

In [ ]:
# filter out only the necessary years - from 2000 to 2018

df = df[(df['Year'] >= 2000) & (df['Year'] <= 2018)]
df2 = df2[(df2['Year'] >= 2000) & (df2['Year'] <= 2018)]

print(len(df))
print(len(df2))

In [ ]:
# sort dates from 2000 to 2018 ascending

df = df.sort_values(by='Incident_Date', ascending=True)
df2 = df2.sort_values(by='Incident_Date', ascending=True)

In [ ]:
df.head(2)

In [ ]:
df2.head(2)

#### Merge Datasets
To merge, we have to find common unique combinations since the date is not unique. There are multiple entries per single date

In [ ]:
print(len(df))          # 5283
print(len(df2))         # 3954

In [ ]:
# to check for duplicaions that are not null

keys = ['Incident_Date', 'Aircaft_Registration', 'Aircaft_Model']

dup_df  = df[df[keys].notna().all(axis=1) & df[keys].duplicated(keep=False)]
dup_df2 = df2[df2[keys].notna().all(axis=1) & df2[keys].duplicated(keep=False)]

print(len(dup_df))
print(len(dup_df2))

In [ ]:
dupes = df2[df2.duplicated(['Incident_Date', 'Aircaft_Registration', 'Aircaft_Model'], keep=False)]
print(len(dupes))
dupes

In [ ]:
# drop the duplicating observation from df2
df2 = df2.drop(index=3393).reset_index(drop=True)
dupes = df2[df2.duplicated(['Incident_Date', 'Aircaft_Registration', 'Aircaft_Model'], keep=False)]
print(len(dupes))
dupes

In [ ]:
dupes = df[df.duplicated(['Incident_Date', 'Aircaft_Registration', 'Aircaft_Model'], keep=False)]
print(len(dupes))
dupes

In [ ]:
# drop the duplicating observation from df2
df = df.drop(index=74	).reset_index(drop=True)
df = df.drop(index=606	).reset_index(drop=True)
df = df.drop(index=1181	).reset_index(drop=True)
df = df.drop(index=3081	).reset_index(drop=True)
df = df.drop(index=4196	).reset_index(drop=True)
dupes = df2[df2.duplicated(['Incident_Date', 'Aircaft_Registration', 'Aircaft_Model'], keep=False)]
print(len(dupes))
dupes

i decided to clean the df2 and merge in any columns from df that makes sense

In [ ]:
df2.columns

In [ ]:
# Incident_Category
# Aircraft_Phase
# Fatalities
# Ground_Casualties
# Collision_Casualties
# Incident_Cause(es)

In [ ]:
df.columns

In [ ]:
print(len(df))
print(len(df2))

In [ ]:
print(df['Incident_Date'].isna().sum())
print(df2['Incident_Date'].isna().sum())
print(df['Aircaft_Registration'].isna().sum())
print(df2['Aircaft_Registration'].isna().sum())

In [ ]:
########################## This code is just a sanity check #######################################

# here we check for entries with incident date and aircraft registration and we get all datasets that have matching entries.
key_cols = ['Incident_Date', 'Aircaft_Registration']

# 2. Find common (non-key) columns
common_cols = df.columns.intersection(df2.columns).tolist()
value_cols  = [c for c in common_cols if c not in key_cols]

# 3. Build a merged dataframe on the keys
merged = pd.merge(
    df[key_cols + value_cols],
    df2[key_cols + value_cols],
    on=key_cols,
    how='inner',
    suffixes=('_df', '_df2')
)

print("Common key matches:", len(merged))

# 4. Loop over each common feature and show comparisons
n = 10  # how many random entries per feature

for col in value_cols:
    col_df  = f'{col}_df'
    col_df2 = f'{col}_df2'

    sub = merged[key_cols + [col_df, col_df2]].copy()

    if sub.empty:
        continue

    # random sample up to n rows
    sub = sub.sample(n=min(n, len(sub)), random_state=0)

    # rename for display
    sub = sub.rename(columns={col_df: 'df', col_df2: 'df2'})

    print(f'\nFeature: {col}')
    print(sub.to_string(index=False))


In [ ]:
# we take different entries from different datasets

# first we only consider entries where 'Incident_Date' and 'Aircaft_Registration' is non-nan in df2 and we do this:

# create a copy of the df as new_df

# overwrite entires of columns new_df 'Aircraft_Phase', 'Incident_Category' in df that match the unique 'Incident_Date' and 'Aircaft_Registration' combination
# add new columns 'Incident_Cause(es)', 'Ground_Casualties', 'Collision_Casualties', to the new dataframe from df2 based on unique combinations of 'Incident_Date' and 'Aircaft_Registration'

# leave the following columns of new_df without overwriting, unless the entries are nan
# 'Year', 'Time', 'Event_No', 'Aircaft_Damage_Type', 'Incident_Location', 'Aircaft_Nature', 'Fatalities', 'Aircaft_Model', 'Aircaft_Operator', 'Engine_Model', 'Departure_Airport', 'Destination_Airport',
# 'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating', 'Narrative' stays as it is in df

new_df = df.copy()

key_cols = ['Incident_Date', 'Aircaft_Registration']

df2_unique_combos = (df2.dropna(subset=key_cols).sort_values(key_cols).drop_duplicates(subset=key_cols, keep='first')) # ensure unique combos

print(len(df2_unique_combos))
print(len(df2))
print(len(df2) - len(df2_unique_combos))
print(len(df2[df2['Incident_Date'].isna() | df2['Aircaft_Registration'].isna()]))

In [ ]:
new_df.head(3)

In [ ]:
# add new columns from df2
cols_to_add = ['Incident_Cause(es)', 'Ground_Casualties', 'Collision_Casualties']

# keep only key + those columns from df2_unique_combos
df2_subset = df2_unique_combos[key_cols + cols_to_add]

print("len new_df before merge: ", len(new_df))

# left-join: keep all rows from new_df, add matching info from df2_unique_combos
new_df = new_df.merge(df2_subset, on=key_cols, how='left')

print("len new_df after merge: ", len(new_df))

new_df.head(3)

In [ ]:
# cols to overwrite from df2 : 'Aircraft_Phase', 'Incident_Category'
for col_to_overwrite in ['Aircraft_Phase', 'Incident_Category']:
    if col_to_overwrite not in df2_unique_combos.columns:
        continue
    df2_subset_overwrite = (
        df2_unique_combos[key_cols + [col_to_overwrite]]
        .dropna(subset=[col_to_overwrite])
    )
    new_df = new_df.merge(
        df2_subset_overwrite,
        on=key_cols,
        how='left',
        suffixes=('', '_df2')
    )
    new_df[col_to_overwrite] = new_df[f'{col_to_overwrite}_df2'].combine_first(new_df[col_to_overwrite])
    new_df = new_df.drop(columns=[f'{col_to_overwrite}_df2'])

print("len new_df after merge: ", len(new_df))

new_df.head(3)

In [ ]:
# if any of these values are missing, fill them

fill_cols = [
    'Time', 'Aircaft_Damage_Type', 'Incident_Location',
    'Aircaft_Nature', 'Fatalities', 'Aircaft_Model', 'Aircaft_Operator',
    'Engine_Model', 'Departure_Airport', 'Destination_Airport',
    'Aircraft_Manufacture_Year'
]

fill_cols_existing = [c for c in fill_cols if c in df2_unique_combos.columns]

df2_fill = df2_unique_combos[key_cols + fill_cols_existing].copy()

new_df = new_df.merge(
    df2_fill,
    on=key_cols,
    how='left',
    suffixes=('', '_df2')
)

for col in fill_cols_existing:
    col_df2 = f'{col}_df2'
    if col_df2 in new_df.columns:
        # keep existing values in new_df[col], fill only NaNs from df2
        new_df[col] = new_df[col].combine_first(new_df[col_df2])

helper_cols = [c for c in new_df.columns if c.endswith('_df2')]
new_df = new_df.drop(columns=helper_cols)

len(new_df)

In [ ]:
new_df.head(10)

In [ ]:
pattern = r'Fatalities:\s*(\d*)\s*/?\s*[\n\r]*\s*Occupants:\s*(\d*)'

df_extract = new_df['Fatalities'].astype(str).str.extract(pattern)
df_extract = df_extract.replace('', np.nan).astype(float)
df_extract.columns = ['Fatalities_num', 'Occupants_num']

new_df['Fatalities_clean'] = df_extract['Fatalities_num']
new_df['Occupants'] = df_extract['Occupants_num']

new_df = new_df.drop(columns=['Fatalities'])

new_df = new_df.rename(columns={'Fatalities_clean': 'Fatalities'})

new_df.head(10)

In [ ]:
new_df.columns

Dealing with Ground casualities and collision casualities

In [ ]:
new_df['Ground_Casualties'] = (
    new_df['Ground_Casualties']
    .str.extract(r'(\d+)')   # extract numbers
    .astype(float)           # convert to float
    .fillna(0)               # replace NaN with 0
)

new_df['Collision_Casualties'] = (
    new_df['Collision_Casualties']
    .str.extract(r'(\d+)')   # extract numbers
    .astype(float)           # convert to float
    .fillna(0)               # replace NaN with 0
)

# All Ground_Casualties and Collision_Casualties are now fatality values

In [ ]:
new_df.head(10)

In [ ]:
new_df.columns

Add lat and lon of airports

In [ ]:
! pip install airportsdata

In [ ]:
import airportsdata

new_df['Departure_Airport_Code'] = new_df['Departure_Airport'].str.extract(r'\((.*?)\)')
new_df['Departure_Airport'] = new_df['Departure_Airport'].str.replace(r'\([^)]*\)', '').str.strip()
new_df.replace('unknown airport', np.nan, inplace = True)
new_df['Destination_Airport_Code'] = new_df['Destination_Airport'].str.extract(r'\((.*?)\)')
new_df['Destination_Airport'] = new_df['Destination_Airport'].str.replace(r'\([^)]*\)', '').str.strip()
new_df['Departure_IATA'] = new_df['Departure_Airport_Code'].str.extract(r'([A-Z]{3})')
new_df['Destination_IATA'] = new_df['Destination_Airport_Code'].str.extract(r'([A-Z]{3})')
new_df[['Departure_Airport', 'Departure_Airport_Code', 'Destination_Airport', 'Destination_Airport_Code', 'Departure_IATA', 'Destination_IATA']].head()

In [ ]:
airports = airportsdata.load('IATA')

def get_airport_info(code, info_type):
    info = airports.get(code)
    return info[info_type] if info else np.nan

new_df['Dep_Lat'] = new_df['Departure_IATA'].map(lambda x: get_airport_info(x, 'lat'))
new_df['Dep_Lon'] = new_df['Departure_IATA'].map(lambda x: get_airport_info(x, 'lon'))

new_df['Dest_Lat'] = new_df['Destination_IATA'].map(lambda x: get_airport_info(x, 'lat'))
new_df['Dest_Lon'] = new_df['Destination_IATA'].map(lambda x: get_airport_info(x, 'lon'))

new_df.head()

# Analysis

In [ ]:
new_df.columns

In [ ]:
new_df['Year'].isna().sum()

In [ ]:
new_df['Aircraft_Phase'].unique()

In [ ]:
canonical_map = {
    'Unknown': 'Unknown',
    'Unknown (UNK)': 'Unknown',

    'Standing': 'Standing',
    'Standing (STD)': 'Standing',

    'Pushback / towing': 'Ground Handling',
    'Pushback / towing (PBT)': 'Ground Handling',

    'Taxi': 'Taxi',
    'Taxi (TXI)': 'Taxi',

    'Takeoff (TOF)': 'Takeoff',
    'Take off': 'Takeoff',

    'Initial climb': 'Initial Climb',
    'Initial climb (ICL)': 'Initial Climb',

    'En route': 'En Route',
    'En route (ENR)': 'En Route',

    'Approach (APR)': 'Approach',
    'Approach': 'Approach',

    'Landing': 'Landing',
    'Landing (LDG)': 'Landing',

    'Manoeuvring (airshow, firefighting, ag.ops.)': 'Maneuvering',
    'Maneuvering (MNV)': 'Maneuvering',
}


new_df['Aircraft_Phase'] = (
    new_df['Aircraft_Phase']
    .replace(canonical_map)
    .fillna('Unknown')     # optional
)

new_df['Aircraft_Phase'].unique()

In [ ]:
new_df['Incident_Category'].unique()

In [ ]:
new_df['Incident'] = new_df['Incident_Category'].str.split(r'\s*\|\s*', n=1).str[0].replace('', np.nan)
new_df['Damage'] = new_df['Incident_Category'].str.split(r'\s*\|\s*', n=1).str[1].replace('', np.nan)
new_df = new_df.drop(columns=['Incident_Category'])

print(new_df['Incident'].unique())

In [ ]:
incident_group_map = {
    'Accident': 'Accident',
    'Serious incident': 'Serious Incident',
    'Incident': 'Incident',

    'Hijacking': 'Hijacking',
    'Unlawful Interference': 'Hijacking',

    'Criminal occurrence (sabotage, shoot down)': 'Criminal',

    'other occurrence (ground fire, sabotage)': 'Other Occurrence',
    'Other': 'Other Occurrence',

    'occurrence unknown': 'Unknown',
    'UK': 'Unknown',
    'Unknown': 'Unknown'
}

new_df['Incident'] = new_df['Incident'].fillna('Unknown')
new_df['Incident'] = new_df['Incident'].map(incident_group_map).fillna('Unknown')

print(new_df['Incident'].unique())

In [ ]:
print(new_df['Damage'].unique())

In [ ]:
new_df['Aircaft_Damage_Type'].unique()

In [ ]:
# fill NaNs in Aircaft_Damage_Type using Damage where available
mask = new_df['Aircaft_Damage_Type'].isna() & new_df['Damage'].notna()
new_df.loc[mask, 'Aircaft_Damage_Type'] = new_df.loc[mask, 'Damage']

new_df = new_df.drop(columns=['Damage'])

new_df['Aircaft_Damage_Type'].unique()

In [ ]:
new_df['Aircaft_Damage_Type'] = new_df['Aircaft_Damage_Type'].fillna('Unknown')

damage_map = {
    'Substantial': 'Substantial',
    'Substantial, repaired': 'Repairable',
    'Substantial, written off': 'Destroyed',

    'Minor': 'Minor',
    'Minor, repaired': 'Repairable',
    'Minor, written off': 'Destroyed',

    'repairable-damage': 'Repairable',

    'Destroyed': 'Destroyed',
    'Destroyed, written off': 'Destroyed',

    'Aircraft missing': 'Missing',
    'Aircraft missing, written off': 'Missing',

    'None, repaired': 'Repairable',

    'Unknown': 'Unknown',
    'Unknown, repaired': 'Repairable',
    'Unknown, written off': 'Unknown'
}

new_df['Aircaft_Damage_Type'] = (
    new_df['Aircaft_Damage_Type']
    .map(damage_map)
    .fillna('Unknown')
)

new_df['Aircaft_Damage_Type'].unique()

In [ ]:
new_df['Aircaft_Nature'].unique()

In [ ]:
nature_map = {
    'Passenger': 'Passenger',
    'Passenger - Scheduled': 'Passenger',
    'Passenger - Non-Scheduled/charter/Air Taxi': 'Passenger',
    'Executive': 'Passenger',

    'Cargo': 'Cargo',

    'Private': 'Private',

    'Military': 'Military',

    'Training': 'Training/Test',
    'Test': 'Training/Test',
    'Calibration/Inspection': 'Training/Test',
    'Demo/Airshow/Display': 'Training/Test',

    'Parachuting': 'Special Operations',
    'Agricultural': 'Special Operations',
    'Fire fighting': 'Special Operations',
    'External load operation': 'Special Operations',
    'Aerial patrol': 'Special Operations',
    'Survey': 'Special Operations',
    'SF': 'Special Operations',   # likely "special flight"
    'Ambulance': 'Special Operations',

    'Ferry/positioning': 'Ferry/Positioning',

    'Illegal Flight': 'Illegal/Unauthorized',

    '-': 'Unknown',
    'Unknown': 'Unknown',
    np.nan: 'Unknown'
}

new_df['Aircaft_Nature'] = new_df['Aircaft_Nature'].fillna('Unknown')
new_df['Aircaft_Nature'] = new_df['Aircaft_Nature'].map(nature_map).fillna('Unknown')

new_df['Aircaft_Nature'].unique()

In [ ]:
new_df

In [ ]:
new_df.columns

In [ ]:
new_df['Incident_Location'].isna().sum()

In [ ]:
new_df['Departure_Airport'].isna().sum()

In [ ]:
df['Departure_Airport'].isna().sum()

In [ ]:
df2['Departure_Airport'].isna().sum()

In [ ]:
new_df.to_csv('new_clean_aircraft_data.csv')

In [ ]:
new_df['Year'].unique()

### Try to map locations

In [ ]:
new_df_copy = new_df.copy()

new_df_copy['Incident_Location'].head(20)

In [ ]:
s = new_df_copy["Incident_Location"]
new_df_copy["Incident_Location"] = (
    s.astype(str)
     .str.encode("latin1", errors="ignore")
     .str.decode("utf-8", errors="ignore")
)

new_df_copy[["raw_place", "country"]] = (
    new_df_copy["Incident_Location"]
      .str.rsplit(" -", n=1, expand=True)
)

new_df_copy["raw_place"] = new_df_copy["raw_place"].str.strip()
new_df_copy["country"]   = new_df_copy["country"].str.strip()

In [ ]:
import re

def build_geopy_query(row):
    place_raw = row.get("Incident_Location", np.nan)
    country   = row.get("country", np.nan)

    # ---------- clean_place logic ----------
    if not isinstance(place_raw, str):
        place = ""
    else:
        s = place_raw.strip()

        # keep part after " of "
        if " of " in s:
            s = s.split(" of ")[-1]

        # remove leading near/over/etc.
        s = re.sub(r"^(near|over|off|approx\.?|around)\s+", "", s, flags=re.I)

        # drop leading distance expressions like "5 km off "
        s = re.sub(
            r"^[0-9.,]+\s*(km|nm|mi|miles)?\s*(N|S|E|W|NE|NW|SE|SW)?\s*",
            "",
            s,
            flags=re.I,
        )

        # remove airport codes in parentheses: (YUL)
        s = re.sub(r"\([^)]*\)", "", s)

        # collapse spaces and trim punctuation
        s = re.sub(r"\s+", " ", s)
        place = s.strip(" ,;-")

    # ---------- make_query logic ----------
    query = np.nan
    if place:
        if isinstance(country, str) and country and country.lower() != "unknown country":
            query = f"{place}, {country}"
        else:
            query = place
    elif isinstance(country, str) and country and country.lower() != "unknown country":
        query = country

    # ---------- fix_failed_query logic ----------
    if pd.isna(query):
        return np.nan

    s = str(query).strip()
    if not s:
        return np.nan

    s_lower = s.lower()

    # remove leading "from "
    if s_lower.startswith("from "):
        s = s[5:].strip()
        s_lower = s.lower()

    # remove leading "off "
    if s_lower.startswith("off "):
        s = s[4:].strip()
        s_lower = s.lower()

    # remove things like "W off", "SE off", etc.
    s = re.sub(r"^[NSEW]{1,2}\s+off\s+", "", s, flags=re.I)

    # remove leading "within "
    if s_lower.startswith("within "):
        s = s[7:].strip()
        s_lower = s.lower()

    # handle "ca 3 km from ..." or "ca 3 km SE off ..."
    s = re.sub(r"^ca\s+[^,]*?\s+(from|off)\s+", "", s, flags=re.I)

    # handle "en route DUS-MUC, Germany" → just country
    s_lower = s.lower()
    if s_lower.startswith("en route "):
        parts = s.split(",")
        if len(parts) > 1:
            s = parts[-1].strip()
        else:
            s = s.replace("en route", "").strip()

    # final cleanup
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" ,;-")

    return s if s else np.nan

# apply to your dataframe
new_df_copy["geopy_query"] = new_df_copy.apply(build_geopy_query, axis=1)

In [ ]:
! pip install geopy

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="aircraft_incidents_locations")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # be nice to the API

In [ ]:
cache = {}

def geocode_cached(query):
    q = str(query).strip() if isinstance(query, str) else None

    # Skip
    if not q:
        status = "SKIPPED"
        print(f"{status} → {query}")
        return pd.Series([np.nan, np.nan, status])

    # Cache hit
    if q in cache:
        lat, lon, status = cache[q]
        print(f"CACHE HIT → {q} → {lat}, {lon}")
        return pd.Series([lat, lon, status])

    # Try geocoding
    try:
        location = geocode(q)
        if location is not None:
            lat, lon = location.latitude, location.longitude
            status = "SUCCESS"
            cache[q] = (lat, lon, status)
            print(f"{status} → {q} → {lat}, {lon}")
            return pd.Series([lat, lon, status])
        else:
            status = "FAILED"
            print(f"{status} → {q}")
    except Exception as e:
        status = f"ERROR"
        print(f"{status} → {q} ({e})")

    cache[q] = (np.nan, np.nan, status)
    return pd.Series([np.nan, np.nan, status])


new_df_copy[["Latitude", "Longitude", "GeoStatus"]] = new_df_copy["geopy_query"].apply(geocode_cached)

In [ ]:
new_df_copy.head()

# we try to solve the lat long problem

In [ ]:
failed_df = new_df_copy[new_df_copy["GeoStatus"] == "FAILED"]
success_df = new_df_copy[new_df_copy["GeoStatus"] == "SUCCESS"]
skipped_df = new_df_copy[new_df_copy["GeoStatus"] == "SKIPPED"]
error_df = new_df_copy[new_df_copy["GeoStatus"] == "ERROR"]

print(len(failed_df))
print(len(success_df))
print(len(skipped_df))
print(len(error_df))

# we have almost 2000 failed queries - problem for later

1957
3321
0
0


In [ ]:
# new_df_copy.to_csv('new_clean_aircraft_data_with_lat_lon_incident_with_failures.csv')

# trying to handle the coordinate failure problem

failed_queries = (
    failed_df["geopy_query"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

failed_queries.to_csv("failed_geopy_queries.txt", index=False, header=False)

In [ ]:
new_df_copy.head()

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

correction_file = list(uploaded.keys())[0]   # get the only uploaded file name

import json

with open(correction_file, "r", encoding="utf-8") as f:
    correction_map = json.load(f)

mask = (new_df_copy["GeoStatus"] == "FAILED")
new_df_copy.loc[mask, "geopy_query"] = (
    new_df_copy.loc[mask, "geopy_query"].map(correction_map).fillna(new_df_copy.loc[mask, "geopy_query"])
)

Saving geopy_corrections.json to geopy_corrections (2).json


In [ ]:
to_retry = new_df_copy["GeoStatus"] == "FAILED"

result = new_df_copy.loc[to_retry, "geopy_query"].apply(geocode_cached)

# assign by position, not by column labels
new_df_copy.loc[to_retry, ["Latitude", "Longitude", "GeoStatus"]] = result.values

CACHE HIT → Nnamdi Azikiwe International Airport, Nigeria → 9.0207137, 7.2677714
SUCCESS → José Joaquín de Olmedo International Airport, Guayaquil, Ecuador → -2.1579327, -79.8833013
CACHE HIT → Félix-Houphouët-Boigny International Airport, Côte d'Ivoire → 5.2649283, -3.9259573
FAILED → Filitheyo Island Seaplane Base, Maldives
CACHE HIT → Aeroparque Jorge Newbery, Buenos Aires, Argentina → -34.5594554, -58.4143637
SUCCESS → Quatro de Fevereiro Airport, Luanda, Angola → -8.861408, 13.2287561
FAILED → Sacramento Mather Airport, California, United States of America


FAILED → El Tornillo area airstrip, Colombia
CACHE HIT → O'Hare International Airport, Chicago, Illinois, United States of America → 41.9782523, -87.9092355


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 1430, in getresponse
    response.begin()
  File "/usr/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ssl.py", line

SUCCESS → Salgado Filho International Airport, Porto Alegre, Rio Grande do Sul, Brazil → -29.9952684, -51.1663978
SUCCESS → Barnstable Municipal Airport, Hyannis, Massachusetts, United States of America → 41.6707658, -70.2843904
FAILED → London Heathrow Airport (Terminal 2, Stand F8), United Kingdom
SUCCESS → Goma International Airport, Goma, Democratic Republic of the Congo → -1.6648635, 29.2381116
FAILED → Montréal–Pierre Elliott Trudeau International Airport, Quebec, Canada
CACHE HIT → Berlin Brandenburg Airport, Germany → 52.3659284, 13.4886435
CACHE HIT → Ingeniero Ambrosio Taravella International Airport, Córdoba, Argentina → nan, nan
CACHE HIT → José Joaquín de Olmedo International Airport, Guayaquil, Ecuador → -2.1579327, -79.8833013
FAILED → N'Djili International Airport, Kinshasa, Democratic Republic of the Congo
CACHE HIT → N'Djili International Airport, Kinshasa, Democratic Republic of the Congo → nan, nan
CACHE HIT → N'Djili International Airport, Kinshasa, Democratic Repu

SUCCESS → Diori Hamani International Airport, Niamey, Niger → 13.4901663, 2.1846007


...

SUCCESS → Southwest Wyoming Regional Airport, Rock Springs, Wyoming, United States of America → 41.5950759, -109.0684214
SUCCESS → Trabzon Airport, Türkiye → 40.9944424, 39.7890385
CACHE HIT → Mitiga International Airport, Tripoli, Libya → 32.8951353, 13.2809484
CACHE HIT → Zhetygen Air Base, Kazakhstan → nan, nan
FAILED → Mwanza International Airport, Tanzania
FAILED → Hodulluca, Yalvaç District, Turkey
CACHE HIT → Berlin Brandenburg Airport, Germany → 52.3659284, 13.4886435
CACHE HIT → Nnamdi Azikiwe International Airport, Nigeria → 9.0207137, 7.2677714
CACHE HIT → Zhengchang, Suiyang County, Guizhou Province, China → nan, nan
CACHE HIT → Sydney Kingsford Smith International Airport, New South Wales, Australia → nan, nan
CACHE HIT → Burke Lakefront Airport, Cleveland, Ohio, United States of America → 41.5188169, -81.6811324
CACHE HIT → Spain (Canaries airspace) → nan, nan
FAILED → Stepanovskoye, Ramenskoye District, Russia
CACHE HIT → Murtala Muhammed International Airport, Lagos, Ni

SUCCESS → Kirksville Regional Airport, Missouri, United States of America → 40.0924817, -92.5432908
CACHE HIT → Lubumbashi International Airport, Democratic Republic of the Congo → -11.5926492, 27.5259188
CACHE HIT → Benito Juárez International Airport, Mexico City, Mexico → 19.4342349, -99.0733121


SUCCESS → Tezpur Air Force Station, Assam, India → 26.7139851, 92.7855981


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 1430, in getresponse
    response.begin()
  File "/usr/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ssl.py", line

FAILED → Hastings Airport, Freetown, Sierra Leone
CACHE HIT → Wadi Bu al Hashm, Libya → nan, nan
CACHE HIT → El Dorado International Airport, Bogotá, Colombia → 4.7020946, -74.1477132
CACHE HIT → Alek Airstrip, South Sudan → nan, nan
CACHE HIT → Mano Dayak International Airport, Agadez, Niger → 16.9709219, 8.0140235
CACHE HIT → Mano Dayak International Airport, Agadez, Niger → 16.9709219, 8.0140235
CACHE HIT → Bunia Airport, Democratic Republic of the Congo → 1.5666587, 30.2172709
CACHE HIT → Simón Bolívar International Airport, Caracas, Venezuela → nan, nan
CACHE HIT → Zarafshan Airport, Uzbekistan → nan, nan
CACHE HIT → Sidra Airport, Libya → 30.6449322, 18.3190161
CACHE HIT → Shabunda Airport, Democratic Republic of the Congo → -2.6897975, 27.3442817
CACHE HIT → José Martí International Airport, Havana, Cuba → 22.9873849, -82.4142531


In [ ]:
failed_df = new_df_copy[new_df_copy["GeoStatus"] == "FAILED"]
success_df = new_df_copy[new_df_copy["GeoStatus"] == "SUCCESS"]
skipped_df = new_df_copy[new_df_copy["GeoStatus"] == "SKIPPED"]
error_df = new_df_copy[new_df_copy["GeoStatus"] == "ERROR"]

print(len(failed_df))
print(len(success_df))
print(len(skipped_df))
print(len(error_df))

807
4471
0
0


In [ ]:
new_df_copy.head()

,Unnamed: 0,Event_No,Incident_Date,Year,Time,Aircaft_Damage_Type,Aircraft_Phase,Incident_Location,Aircaft_Nature,Aircaft_Registration,...,Dest_Lat,Dest_Lon,Incident,raw_place,country,place_clean,geopy_query,Latitude,Longitude,GeoStatus
0,0,1,2000-01-01,2000,13:00 LT,Substantial,Unknown,"HOMESTEAD, Florida - United States of America",Ferry/Positioning,N752CC,...,26.17100,56.24060,Accident,"HOMESTEAD, Florida",United States of America,"HOMESTEAD, Florida","HOMESTEAD, Florida, United States of America",25.471895,-80.475990,SUCCESS
1,1,2,2000-01-03,2000,NaN,Destroyed,Unknown,unknown location - Unknown country,Unknown,A2-AEZ,...,NaN,NaN,Unknown,unknown location,Unknown country,unknown location,unknown location,-0.999945,36.741661,SUCCESS
2,2,3,2000-01-04,2000,17:25 LT,Substantial,Unknown,"JACKSON, Wyoming - United States of America",Private,N895TT,...,56.17290,92.49330,Accident,"JACKSON, Wyoming",United States of America,"JACKSON, Wyoming","JACKSON, Wyoming, United States of America",43.479965,-110.761815,SUCCESS
3,3,4,2000-01-05,2000,13:25,Destroyed,Approach,Abuja International Airport (ABV) - Nigeria,Passenger,5N-AXL,...,9.00679,7.26317,Accident,Abuja International Airport (ABV),Nigeria,Abuja International Airport,"Nnamdi Azikiwe International Airport, Nigeria",9.020714,7.267771,SUCCESS
4,4,5,2000-01-07,2000,NaN,Missing,Unknown,- Angola,Cargo,D2-FBR,...,-8.78361,17.98970,Accident,- Angola,NaN,Angola,Angola,-11.877577,17.569124,SUCCESS


In [ ]:
new_df_copy.to_csv('new_clean_aircraft_data_with_lat_lon_incident_with_failures.csv')

## NLP to extract keywords regarding incident nature - for analytics

In [ ]:
# this code is working. but try better prompts --- cause the current ones are pretty bad

In [ ]:
import pandas as pd
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA A100-SXM4-80GB


In [ ]:
url = "https://raw.githubusercontent.com/oshani-jayawardane/aircraft-accident-project/main/cleanest_df_with_NLP.csv"
clean_df = pd.read_csv(url)

clean_df.head()

,Unnamed: 0.1,Unnamed: 0,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,...,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,cause,name,is_cause
0,0,0,1,2000-01-01,2000,13:00 LT,Accident,Substantial,Unknown,3.0,...,0.0,0.0,0.0,1.0,0.0,0.0,['right wing leading edge'],NaN,NaN,NaN
1,1,1,2,2000-01-03,2000,NaN,Unknown,Destroyed,Unknown,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,[],NaN,NaN,NaN
2,2,2,3,2000-01-04,2000,17:25 LT,Accident,Substantial,Unknown,3.0,...,0.0,1.0,1.0,0.0,1.0,0.0,['nose landing gear'],NaN,NaN,NaN
3,3,3,4,2000-01-05,2000,13:25,Accident,Destroyed,Approach,13.0,...,0.0,0.0,0.0,0.0,1.0,0.0,['farmland'],NaN,NaN,NaN
4,4,4,5,2000-01-07,2000,NaN,Accident,Missing,Unknown,8.0,...,0.0,0.0,0.0,0.0,0.0,1.0,[],NaN,NaN,NaN


In [ ]:
clean_df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Event_No', 'Incident_Date', 'Year',
       'Time', 'Incident', 'Aircaft_Damage_Type', 'Aircraft_Phase',
       'Occupants', 'Fatalities', 'Ground_Casualties', 'Collision_Casualties',
       'Incident_Location', 'geopy_query', 'Latitude', 'Longitude',
       'GeoStatus', 'Aircaft_Nature', 'Aircaft_Registration', 'Aircaft_Model',
       'Aircaft_Operator', 'Engine_Model', 'Departure_Airport',
       'Destination_Airport', 'Departure_Airport_Code',
       'Destination_Airport_Code', 'Departure_IATA', 'Destination_IATA',
       'Dep_Lat', 'Dep_Lon', 'Dest_Lat', 'Dest_Lon',
       'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating',
       'Narrative', 'Incident_Cause(es)', 'is_engine_cause', 'is_model_cause',
       'is_human_cause', 'is_weather_cause', 'is_wildlife_cause',
       'is_ground_collision', 'is_unknown_cause', 'damage_area', 'cause',
       'name', 'is_cause'],
      dtype='object')

In [ ]:
clean_df.iloc[3344:3352]

,Unnamed: 0.1,Unnamed: 0,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,...,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,cause,name,is_cause
3344,3344,3344,3393,2011-08-25,2011,NaN,Criminal,Destroyed,Standing,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,['lower fuselage'],NaN,NaN,NaN
3345,3345,3345,3394,2011-08-26,2011,NaN,Hijacking,Destroyed,Standing,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,[],NaN,NaN,NaN
3346,3346,3346,3395,2011-08-28,2011,08:54,Accident,Destroyed,Maneuvering,2.0,...,0.0,1.0,0.0,0.0,1.0,0.0,['fuselage'],NaN,NaN,NaN
3347,3347,3347,3396,2011-08-29,2011,03:54,Accident,Repairable,Landing,143.0,...,0.0,1.0,1.0,0.0,1.0,0.0,['nose gear'],NaN,NaN,NaN
3348,3348,3348,3397,2011-08-31,2011,NaN,Accident,Substantial,Landing,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3349,3349,3349,3398,2011-09-02,2011,17:48,Accident,Destroyed,Approach,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3350,3350,3350,3399,2011-09-02,2011,13:35,Accident,Destroyed,En Route,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3351,3351,3351,3400,2011-09-03,2011,12:00,Accident,Destroyed,Maneuvering,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
start_idx = 3348   # first row that needs NLP

processed_df = clean_df.iloc[:start_idx].reset_index(drop=True)      # 0..3347
new_df_source = clean_df.iloc[start_idx:].reset_index(drop=True)     # 3348..end

processed_df.head()

,Unnamed: 0.1,Unnamed: 0,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,...,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,cause,name,is_cause
0,0,0,1,2000-01-01,2000,13:00 LT,Accident,Substantial,Unknown,3.0,...,0.0,0.0,0.0,1.0,0.0,0.0,['right wing leading edge'],NaN,NaN,NaN
1,1,1,2,2000-01-03,2000,NaN,Unknown,Destroyed,Unknown,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,[],NaN,NaN,NaN
2,2,2,3,2000-01-04,2000,17:25 LT,Accident,Substantial,Unknown,3.0,...,0.0,1.0,1.0,0.0,1.0,0.0,['nose landing gear'],NaN,NaN,NaN
3,3,3,4,2000-01-05,2000,13:25,Accident,Destroyed,Approach,13.0,...,0.0,0.0,0.0,0.0,1.0,0.0,['farmland'],NaN,NaN,NaN
4,4,4,5,2000-01-07,2000,NaN,Accident,Missing,Unknown,8.0,...,0.0,0.0,0.0,0.0,0.0,1.0,[],NaN,NaN,NaN


In [ ]:
new_df_source.head()

,Unnamed: 0.1,Unnamed: 0,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,...,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,cause,name,is_cause
0,3348,3348,3397,2011-08-31,2011,NaN,Accident,Substantial,Landing,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3349,3349,3398,2011-09-02,2011,17:48,Accident,Destroyed,Approach,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3350,3350,3399,2011-09-02,2011,13:35,Accident,Destroyed,En Route,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3351,3351,3400,2011-09-03,2011,12:00,Accident,Destroyed,Maneuvering,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3352,3352,3401,2011-09-04,2011,15:29,Accident,Destroyed,Landing,47.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import json
import math
from tqdm.auto import tqdm

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
# Helper: build input text with narrative priority

def build_text(narrative, cause):
    # Handle NaNs / None
    narr = "" if narrative is None or (isinstance(narrative, float) and math.isnan(narrative)) else str(narrative).strip()
    cause_txt = "" if cause is None or (isinstance(cause, float) and math.isnan(cause)) else str(cause).strip()

    if not narr and not cause_txt:
        return None  # nothing to analyze

    # Narrative first, cause second (narrative has priority)
    text_parts = []
    if narr:
        text_parts.append(f"NARRATIVE:\n{narr}")
    if cause_txt:
        text_parts.append(f"INCIDENT_CAUSE_FIELD:\n{cause_txt}")

    return "\n\n".join(text_parts)


In [ ]:
def build_prompt(case_text):
    prompt = f"""
You are an aviation accident investigator.

Given the information below, decide which factors DIRECTLY CAUSED the accident.
Mentions that did NOT cause the accident must NOT be tagged as causes.
If the narrative and the incident cause field disagree, give MORE WEIGHT to the narrative.

You must answer ONLY in valid JSON (no extra text).

Definitions:
- is_engine_cause: 1 if an internal engine or propulsion system failure directly caused the accident.
- is_model_cause: 1 if a design flaw or known problem with the aircraft model directly caused the accident.
- is_human_cause: 1 if pilot, crew, or staff actions/decisions (carelessness, violation of procedure, etc.) directly caused the accident.
- is_weather_cause: 1 if weather or environmental conditions (storms, icing, turbulence, wind shear, etc.) directly caused the accident.
- is_wildlife_cause: 1 if a bird or animal strike directly caused the accident.
- is_ground_collision: 1 if collision with ground objects (vehicles, buildings, airport equipment, terrain, etc.) directly caused the accident.
- is_unknown_cause: 1 if the true cause cannot be determined from the text.

Multiple causes may be 1 at the same time if the text CLEARLY supports them ONLY.

Also extract:
- damage_area: list of STRUCTURAL aircraft parts that were DAMAGED (e.g. "left wing", "right wing", "fuselage", "wing leading edge", "landing gear", "engine", "engine #1", "propeller", "tail", "nose", "cockpit windows", "vertical stabilizer","horizontal stabilizer", "rudder", "elevator", and more).
  Look for keywords but ONLY list parts that are clearly damaged.

EXAMPLE 1
Text:
"A Cessna 172 lost power after an internal engine failure due to oil starvation and crashed short of the runway. The airplane sustained substantial damage to the nose and landing gear."
Answer:
{{
 "is_engine_cause": 1,
 "is_model_cause": 0,
 "is_human_cause": 0,
 "is_weather_cause": 0,
 "is_wildlife_cause": 0,
 "is_ground_collision": 0,
 "is_unknown_cause": 0,
 "damage_area": ["nose", "landing gear"]
}}

EXAMPLE 2
Text:
"The pilot continued an unstable approach at excessive speed, landed long, and overran the runway, collapsing the nose gear. No mechanical anomalies were reported with the airplane."
Answer:
{{
 "is_engine_cause": 0,
 "is_model_cause": 0,
 "is_human_cause": 1,
 "is_weather_cause": 0,
 "is_wildlife_cause": 0,
 "is_ground_collision": 1,
 "is_unknown_cause": 0,
 "damage_area": ["nose gear"]
}}

EXAMPLE 3
Text:
"The airplane struck a flock of birds during climb, resulting in dents and punctures to the right wing leading edge and engine inlet. The pilot returned for a precautionary landing."
Answer:
{{
 "is_engine_cause": 0,
 "is_model_cause": 0,
 "is_human_cause": 0,
 "is_weather_cause": 0,
 "is_wildlife_cause": 1,
 "is_ground_collision": 0,
 "is_unknown_cause": 0,
 "damage_area": ["right wing leading edge", "engine inlet"]
}}

NOW ANALYZE THIS CASE:
Text:
\"\"\"{case_text}\"\"\"

Return ONLY JSON for this case with exactly these keys:
{{
 "is_engine_cause": 0 or 1,
 "is_model_cause": 0 or 1,
 "is_human_cause": 0 or 1,
 "is_weather_cause": 0 or 1,
 "is_wildlife_cause": 0 or 1,
 "is_ground_collision": 0 or 1,
 "is_unknown_cause": 0 or 1,
 "damage_area": [ "part1", "part2", ... ]
}}
"""
    return prompt.strip()

In [ ]:
import re

def generate_json(case_text, max_new_tokens=256, temperature=0.1):
    prompt = build_prompt(case_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=False
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Try to extract the LAST JSON block (our actual answer)
    try:
        matches = re.findall(r"\{[\s\S]*?\}", full_text)
        if not matches:
            raise ValueError("No JSON object found in model output.")
        json_str = matches[-1]   # take the LAST {...}
        data = json.loads(json_str)
    except Exception as e:
        # print("PARSE ERROR:", e)  # optional debug
        data = {
            "is_engine_cause": 0,
            "is_model_cause": 0,
            "is_human_cause": 0,
            "is_weather_cause": 0,
            "is_wildlife_cause": 0,
            "is_ground_collision": 0,
            "is_unknown_cause": 1,
            "damage_area": []
        }

    # Ensure all keys exist and have sane types
    default = {
        "is_engine_cause": 0,
        "is_model_cause": 0,
        "is_human_cause": 0,
        "is_weather_cause": 0,
        "is_wildlife_cause": 0,
        "is_ground_collision": 0,
        "is_unknown_cause": 0,
        "damage_area": []
    }
    for k, v in default.items():
        if k not in data:
            data[k] = v

    cause_keys = [
        "is_engine_cause",
        "is_model_cause",
        "is_human_cause",
        "is_weather_cause",
        "is_wildlife_cause",
        "is_ground_collision"
    ]
    if sum(int(data.get(k, 0)) for k in cause_keys) == 0 and int(data.get("is_unknown_cause", 0)) == 0:
        data["is_unknown_cause"] = 1

    da = data.get("damage_area", [])
    if not isinstance(da, list):
        da = [str(da)]
    data["damage_area"] = [str(x).strip() for x in da if str(x).strip()]

    return data

In [ ]:
from tqdm.auto import tqdm

results_new = []

for _, row in tqdm(new_df_source.iterrows(), total=len(new_df_source)):
    case_text = build_text(row.get("Narrative"), row.get("Incident_Cause(es)"))

    if case_text is None:
        res = {
            "is_engine_cause": 0,
            "is_model_cause": 0,
            "is_human_cause": 0,
            "is_weather_cause": 0,
            "is_wildlife_cause": 0,
            "is_ground_collision": 0,
            "is_unknown_cause": 1,
            "damage_area": []
        }
    else:
        res = generate_json(case_text)

    results_new.append(res)


  0%|          | 0/1930 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

In [ ]:
results_new_df = pd.DataFrame(results_new)

results_new_df.head()

,is_engine_cause,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,incident_cause_field,is_wake_vortex_cause,is_model_cause_details,probable_cause
0,0,0,0,0,0,1,0,[],NaN,NaN,NaN,NaN
1,0,0,0,1,0,0,0,"[left wing, nose, cockpit, right wing]",NaN,NaN,NaN,NaN
2,0,0,0,0,1,0,0,"[right wing leading edge, engine inlet]",NaN,NaN,NaN,NaN
3,0,0,0,0,0,1,0,[airplane],NaN,NaN,NaN,NaN
4,0,0,1,1,0,1,0,"[nose, landing gear, fuselage]",NaN,NaN,NaN,NaN


In [ ]:
print(len(results_new_df))
print(len(clean_df))

1930
5278


In [ ]:
print(processed_df.shape)
processed_df.head()

(3348, 49)


,Unnamed: 0.1,Unnamed: 0,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,...,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,cause,name,is_cause
0,0,0,1,2000-01-01,2000,13:00 LT,Accident,Substantial,Unknown,3.0,...,0.0,0.0,0.0,1.0,0.0,0.0,['right wing leading edge'],NaN,NaN,NaN
1,1,1,2,2000-01-03,2000,NaN,Unknown,Destroyed,Unknown,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,[],NaN,NaN,NaN
2,2,2,3,2000-01-04,2000,17:25 LT,Accident,Substantial,Unknown,3.0,...,0.0,1.0,1.0,0.0,1.0,0.0,['nose landing gear'],NaN,NaN,NaN
3,3,3,4,2000-01-05,2000,13:25,Accident,Destroyed,Approach,13.0,...,0.0,0.0,0.0,0.0,1.0,0.0,['farmland'],NaN,NaN,NaN
4,4,4,5,2000-01-07,2000,NaN,Accident,Missing,Unknown,8.0,...,0.0,0.0,0.0,0.0,0.0,1.0,[],NaN,NaN,NaN


In [ ]:
processed_df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Event_No', 'Incident_Date', 'Year',
       'Time', 'Incident', 'Aircaft_Damage_Type', 'Aircraft_Phase',
       'Occupants', 'Fatalities', 'Ground_Casualties', 'Collision_Casualties',
       'Incident_Location', 'geopy_query', 'Latitude', 'Longitude',
       'GeoStatus', 'Aircaft_Nature', 'Aircaft_Registration', 'Aircaft_Model',
       'Aircaft_Operator', 'Engine_Model', 'Departure_Airport',
       'Destination_Airport', 'Departure_Airport_Code',
       'Destination_Airport_Code', 'Departure_IATA', 'Destination_IATA',
       'Dep_Lat', 'Dep_Lon', 'Dest_Lat', 'Dest_Lon',
       'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating',
       'Narrative', 'Incident_Cause(es)', 'is_engine_cause', 'is_model_cause',
       'is_human_cause', 'is_weather_cause', 'is_wildlife_cause',
       'is_ground_collision', 'is_unknown_cause', 'damage_area'],
      dtype='object')

In [ ]:
processed_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], inplace=True)
processed_df.shape

(3348, 44)

In [ ]:
processed_df.columns

Index(['Event_No', 'Incident_Date', 'Year', 'Time', 'Incident',
       'Aircaft_Damage_Type', 'Aircraft_Phase', 'Occupants', 'Fatalities',
       'Ground_Casualties', 'Collision_Casualties', 'Incident_Location',
       'geopy_query', 'Latitude', 'Longitude', 'GeoStatus', 'Aircaft_Nature',
       'Aircaft_Registration', 'Aircaft_Model', 'Aircaft_Operator',
       'Engine_Model', 'Departure_Airport', 'Destination_Airport',
       'Departure_Airport_Code', 'Destination_Airport_Code', 'Departure_IATA',
       'Destination_IATA', 'Dep_Lat', 'Dep_Lon', 'Dest_Lat', 'Dest_Lon',
       'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating',
       'Narrative', 'Incident_Cause(es)', 'is_engine_cause', 'is_model_cause',
       'is_human_cause', 'is_weather_cause', 'is_wildlife_cause',
       'is_ground_collision', 'is_unknown_cause', 'damage_area'],
      dtype='object')

In [ ]:
new_df_source.head()

,Unnamed: 0.1,Unnamed: 0,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,...,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,cause,name,is_cause
0,3348,3348,3397,2011-08-31,2011,NaN,Accident,Substantial,Landing,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3349,3349,3398,2011-09-02,2011,17:48,Accident,Destroyed,Approach,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3350,3350,3399,2011-09-02,2011,13:35,Accident,Destroyed,En Route,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3351,3351,3400,2011-09-03,2011,12:00,Accident,Destroyed,Maneuvering,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3352,3352,3401,2011-09-04,2011,15:29,Accident,Destroyed,Landing,47.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
new_df_source.shape

(1930, 49)

In [ ]:
new_df_source.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Event_No', 'Incident_Date', 'Year',
       'Time', 'Incident', 'Aircaft_Damage_Type', 'Aircraft_Phase',
       'Occupants', 'Fatalities', 'Ground_Casualties', 'Collision_Casualties',
       'Incident_Location', 'geopy_query', 'Latitude', 'Longitude',
       'GeoStatus', 'Aircaft_Nature', 'Aircaft_Registration', 'Aircaft_Model',
       'Aircaft_Operator', 'Engine_Model', 'Departure_Airport',
       'Destination_Airport', 'Departure_Airport_Code',
       'Destination_Airport_Code', 'Departure_IATA', 'Destination_IATA',
       'Dep_Lat', 'Dep_Lon', 'Dest_Lat', 'Dest_Lon',
       'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating',
       'Narrative', 'Incident_Cause(es)', 'is_engine_cause', 'is_model_cause',
       'is_human_cause', 'is_weather_cause', 'is_wildlife_cause',
       'is_ground_collision', 'is_unknown_cause', 'damage_area', 'cause',
       'name', 'is_cause'],
      dtype='object')

In [ ]:
new_df_source.drop(columns=['Unnamed: 0.1', 'Unnamed: 0', 'is_engine_cause', 'is_model_cause',
       'is_human_cause', 'is_weather_cause', 'is_wildlife_cause',
       'is_ground_collision', 'is_unknown_cause', 'damage_area', 'cause',
       'name', 'is_cause'], inplace=True)
new_df_source.shape

(1930, 36)

In [ ]:
new_df_source.head()

,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,Fatalities,Ground_Casualties,...,Destination_IATA,Dep_Lat,Dep_Lon,Dest_Lat,Dest_Lon,Aircraft_Manufacture_Year,Total_Flight_Hours,Confidence_Rating,Narrative,Incident_Cause(es)
0,3397,2011-08-31,2011,NaN,Accident,Substantial,Landing,3.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,1989.0,NaN,NaN,An Antonov 2T airplane sustained substantial d...,Info-Unavailable
1,3398,2011-09-02,2011,17:48,Accident,Destroyed,Approach,21.0,21.0,0.0,...,SCI,-33.39300,-70.78580,7.80132,-72.20290,1994.0,NaN,NaN,"A CASA C-212 Aviocar transport plane, operated...","Result - Loss of control, Weather - Windshear/..."
2,3399,2011-09-02,2011,13:35,Accident,Destroyed,En Route,1.0,1.0,0.0,...,BET,60.50000,-165.10000,60.77856,-161.83717,2000.0,8483 hours,Accident investigation report completed and in...,"A Cessna 208B Grand Caravan airplane, N207DR, ...","Collision - Aircraft, Collision - Aircraft - I..."
3,3400,2011-09-03,2011,12:00,Accident,Destroyed,Maneuvering,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,1983.0,6010 hours,"Information is only available from news, socia...",An Antonov 2 airplane was destroyed in an acci...,Info-Unavailable
4,3401,2011-09-04,2011,15:29,Accident,Destroyed,Landing,47.0,0.0,0.0,...,YOW,41.97694,-87.90815,45.32250,-75.66920,2000.0,25655 hours,Accident investigation report completed and in...,An Embraer EMB-145LR passenger jet sustained s...,Result - Runway excursion


In [ ]:
results_new_df.head()

,is_engine_cause,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,incident_cause_field,is_wake_vortex_cause,is_model_cause_details,probable_cause
0,0,0,0,0,0,1,0,[],NaN,NaN,NaN,NaN
1,0,0,0,1,0,0,0,"[left wing, nose, cockpit, right wing]",NaN,NaN,NaN,NaN
2,0,0,0,0,1,0,0,"[right wing leading edge, engine inlet]",NaN,NaN,NaN,NaN
3,0,0,0,0,0,1,0,[airplane],NaN,NaN,NaN,NaN
4,0,0,1,1,0,1,0,"[nose, landing gear, fuselage]",NaN,NaN,NaN,NaN


In [ ]:
results_new_df.columns

Index(['is_engine_cause', 'is_model_cause', 'is_human_cause',
       'is_weather_cause', 'is_wildlife_cause', 'is_ground_collision',
       'is_unknown_cause', 'damage_area', 'incident_cause_field',
       'is_wake_vortex_cause', 'is_model_cause_details', 'probable_cause'],
      dtype='object')

In [ ]:
results_new_df.drop(columns=['incident_cause_field',
       'is_wake_vortex_cause', 'is_model_cause_details', 'probable_cause'], inplace=True)
results_new_df.shape

(1930, 8)

In [ ]:
results_new_df.head()

,is_engine_cause,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area
0,0,0,0,0,0,1,0,[]
1,0,0,0,1,0,0,0,"[left wing, nose, cockpit, right wing]"
2,0,0,0,0,1,0,0,"[right wing leading edge, engine inlet]"
3,0,0,0,0,0,1,0,[airplane]
4,0,0,1,1,0,1,0,"[nose, landing gear, fuselage]"


In [ ]:
augmented_df = pd.concat(
    [new_df_source.reset_index(drop=True),
     results_new_df.reset_index(drop=True)],
    axis=1
)

augmented_df.shape

(1930, 44)

In [ ]:
print(processed_df.columns)

print(augmented_df.columns)

Index(['Event_No', 'Incident_Date', 'Year', 'Time', 'Incident',
       'Aircaft_Damage_Type', 'Aircraft_Phase', 'Occupants', 'Fatalities',
       'Ground_Casualties', 'Collision_Casualties', 'Incident_Location',
       'geopy_query', 'Latitude', 'Longitude', 'GeoStatus', 'Aircaft_Nature',
       'Aircaft_Registration', 'Aircaft_Model', 'Aircaft_Operator',
       'Engine_Model', 'Departure_Airport', 'Destination_Airport',
       'Departure_Airport_Code', 'Destination_Airport_Code', 'Departure_IATA',
       'Destination_IATA', 'Dep_Lat', 'Dep_Lon', 'Dest_Lat', 'Dest_Lon',
       'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating',
       'Narrative', 'Incident_Cause(es)', 'is_engine_cause', 'is_model_cause',
       'is_human_cause', 'is_weather_cause', 'is_wildlife_cause',
       'is_ground_collision', 'is_unknown_cause', 'damage_area'],
      dtype='object')
Index(['Event_No', 'Incident_Date', 'Year', 'Time', 'Incident',
       'Aircaft_Damage_Type', 'Aircraft_Phase',

In [ ]:
print(len(processed_df))
print(len(augmented_df))
print(len(processed_df) + len(augmented_df))
print(len(clean_df))

3348
1930
5278
5278


In [ ]:
final_df = pd.concat(
    [processed_df, augmented_df],
    axis=0,
    ignore_index=True
)

final_df.to_csv("cleanest_df_with_full_NLP.csv", index=False)

In [ ]:
# clean_df = pd.concat([clean_df, results_df], axis=1)
# clean_df.to_csv('cleanest_df_with_NLP.csv')

# Different model starting here

In [ ]:
# !pip install -q transformers accelerate sentencepiece

In [ ]:
# from transformers import pipeline

# # combine cause + narrative into one text field
# def combine_text(row):
#     cause = str(row.get("Incident_Cause(es)", "") or "")
#     narr  = str(row.get("Narrative", "") or "")
#     return (cause + " " + narr).strip()

# new_df_copy["full_text"] = new_df_copy.apply(combine_text, axis=1)

# labels = [
#     "Did an internal engine or propulsion system failure directly cause the accident?",
#     "Did pilot, crew, or staff actions or decisions directly cause the accident?",
#     "Did weather or environmental conditions directly cause the accident?",
#     "Did a mechanical or structural failure of the aircraft (other than engines) directly cause the accident?",
#     "Did the aircraft collide with birds or wildlife?",
#     "Did the aircraft collide with a ground object or airport infrastructure?",
#     "Was the primary cause of the accident unknown or undetermined?",
#     "Did the aircraft experience a runway excursion or veer-off event as a result?",
#     "Did the flight lose control as a result?",
#     "Did emergency landing, forced landing, aborting happened as a result?"
# ]

# # zero-shot classifier (English, good general-purpose)
# classifier = pipeline(
#     "zero-shot-classification",
#     model="facebook/bart-large-mnli",   # or any NLI zero-shot model
#     device_map="auto"                  # uses GPU if available
# )

# def classify_causes(text, threshold=0.35):
#     if not text or text.strip() == "":
#         return {lbl: 0 for lbl in labels}

#     result = classifier(
#         text,
#         candidate_labels=labels,
#         multi_label=True
#     )

#     # result["labels"] are sorted by score
#     scores = dict(zip(result["labels"], result["scores"]))
#     return {lbl: int(scores.get(lbl, 0.0) >= threshold) for lbl in labels}

# # apply to your dataframe
# cause_flags = new_df_copy["full_text"].apply(classify_causes)

# # convert list of dicts -> columns
# flags_df = pd.DataFrame(list(cause_flags))

# # give nicer column names
# rename_map = {
#     "Did an internal engine or propulsion system failure directly cause the accident?": "is_engine_cause",
#     "Did pilot, crew, or staff actions or decisions directly cause the accident?": "is_crew_cause",
#     "Did weather or environmental conditions directly cause the accident?": "is_weather_cause",
#     "Did a mechanical or structural failure of the aircraft (other than engines) directly cause the accident?": "is_model_cause",
#     "Did the aircraft collide with birds or wildlife?": "is_bird_or_animal",
#     "Did the aircraft collide with a ground object or airport infrastructure?": "is_ground_collision",
#     "Was the primary cause of the accident unknown or undetermined?": "is_unknown_cause",
#     "Did the aircraft experience a runway excursion or veer-off event?": "result_runway_excursion",
#     "Did the flight lose control?": "result_lose_control",
#     "Did emergency landing, forced landing, aborting happened as a result?": "result_forced_landing"
# }

# flags_df = flags_df.rename(columns=rename_map)

# # attach back to df
# new_df_copy = pd.concat([new_df_copy, flags_df], axis=1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


# Damage Analysis

In [ ]:
final_df.head()

,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,Fatalities,Ground_Casualties,...,Narrative,Incident_Cause(es),is_engine_cause,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area
0,1,2000-01-01,2000,13:00 LT,Accident,Substantial,Unknown,3.0,0.0,0.0,...,The pilot-in-command (PIC) stated he was in cr...,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,['right wing leading edge']
1,2,2000-01-03,2000,NaN,Unknown,Destroyed,Unknown,NaN,NaN,0.0,...,Damaged beyond repair.,Info-Unavailable,0.0,0.0,0.0,0.0,0.0,0.0,1.0,[]
2,3,2000-01-04,2000,17:25 LT,Accident,Substantial,Unknown,3.0,0.0,0.0,...,While performing the ILS runway 18 approach to...,NaN,0.0,0.0,1.0,1.0,0.0,1.0,0.0,['nose landing gear']
3,4,2000-01-05,2000,13:25,Accident,Destroyed,Approach,13.0,1.0,1.0,...,The Bandeirante aircraft was coming in to land...,Result - Loss of control,0.0,0.0,0.0,0.0,0.0,1.0,0.0,['farmland']
4,5,2000-01-07,2000,NaN,Accident,Missing,Unknown,8.0,NaN,0.0,...,Disappeared near the border of the Angolan pro...,Unknown - Missing,0.0,0.0,0.0,0.0,0.0,0.0,1.0,[]


In [ ]:
final_df.shape

(5278, 44)

In [ ]:
url = "https://raw.githubusercontent.com/oshani-jayawardane/aircraft-accident-project/main/cleanest_df_with_NLP.csv"
new_final_df = pd.read_csv(url)

new_final_df.head()

,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,Fatalities,Ground_Casualties,...,is_model_cause,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,Engine_Model_Standardized,Aircraft_Manufacturer,Engine_Manufacturer
0,1,2000-01-01,2000,13:00 LT,Accident,Substantial,Unknown,3.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,['right wing leading edge'],Pratt & Whitney JT15D-4,Cessna,Pratt & Whitney
1,2,2000-01-03,2000,NaN,Unknown,Destroyed,Unknown,NaN,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,[],NaN,Beechcraft,NaN
2,3,2000-01-04,2000,17:25 LT,Accident,Substantial,Unknown,3.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,['nose landing gear'],Pratt & Whitney PT6-A-42,Beechcraft,Pratt & Whitney
3,4,2000-01-05,2000,13:25,Accident,Destroyed,Approach,13.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,['farmland'],Pratt & Whitney Canada PT6A-34,Embraer,Pratt & Whitney
4,5,2000-01-07,2000,NaN,Accident,Missing,Unknown,8.0,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,[],Ivchenko AI-24,Antonov,Ivchenko


In [ ]:
new_final_df.columns

Index(['Event_No', 'Incident_Date', 'Year', 'Time', 'Incident',
       'Aircaft_Damage_Type', 'Aircraft_Phase', 'Occupants', 'Fatalities',
       'Ground_Casualties', 'Collision_Casualties', 'Incident_Location',
       'geopy_query', 'Latitude', 'Longitude', 'GeoStatus', 'Aircaft_Nature',
       'Aircaft_Registration', 'Aircaft_Model', 'Aircaft_Operator',
       'Engine_Model', 'Departure_Airport', 'Destination_Airport',
       'Departure_Airport_Code', 'Destination_Airport_Code', 'Departure_IATA',
       'Destination_IATA', 'Dep_Lat', 'Dep_Lon', 'Dest_Lat', 'Dest_Lon',
       'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating',
       'Narrative', 'Incident_Cause(es)', 'is_engine_cause', 'is_model_cause',
       'is_human_cause', 'is_weather_cause', 'is_wildlife_cause',
       'is_ground_collision', 'is_unknown_cause', 'damage_area',
       'Engine_Model_Standardized', 'Aircraft_Manufacturer',
       'Engine_Manufacturer'],
      dtype='object')

In [ ]:
new_final_df['damage_area']

,damage_area
0,['right wing leading edge']
1,[]
2,['nose landing gear']
3,['farmland']
4,[]
...,...
5273,[]
5274,[]
5275,"['nose landing gear', 'pressure vessel aft of ..."
5276,['no. 2 engine']


In [ ]:
new_final_df['damage_area'].unique()

array(["['right wing leading edge']", '[]', "['nose landing gear']", ...,
       "['cockpit section']",
       "['nose section', 'avionics', 'fuselage skin', 'wing leading edge']",
       "['nose landing gear', 'pressure vessel aft of gear attach bulkhead']"],
      dtype=object)

In [ ]:
import re

def simplify_damage(entry):
    # entry can be a string or a list of strings
    if isinstance(entry, list):
        text = " ".join(str(x) for x in entry if x is not None)
    else:
        text = "" if entry is None else str(entry)

    t = text.lower().strip()
    if t == "" or t in {"-", "none", "n/a", "n\\a", "unknown", "undetermined", "unreported", "unspecified"}:
        return []

    cats = set()

    # ---------- WING ----------
    if re.search(r"\bwing\b|\bwings\b|\bwingtip\b|\bwing tip\b|\bwinglet\b|\bslat\b|\bslats\b|\baileron\b|\bflap\b|\bflaps\b|\bspar\b|\bmainplane\b", t):
        cats.add("wing")

    # ---------- ENGINE ----------
    if ("engine" in t or "nacelle" in t or "cowling" in t or "pylon" in t or
        "apu" in t or "fan" in t or "turbine" in t or "jet engine" in t or
        "propeller" in t or "props" in t or re.search(r"\bprop\b", t)):
        # folding propeller into engine to keep categories simple
        cats.add("engine")

    # ---------- TAIL ----------
    if ("tail" in t or "empennage" in t or "tailplane" in t or
        "vertical stabilizer" in t or "horizontal stabilizer" in t or
        "stabilizer" in t or "tail cone" in t or "tail skid" in t or
        "fin" in t or "rudder" in t or "elevator" in t or "tail boom" in t):
        cats.add("tail")

    # ---------- NOSE ----------
    if ("nose" in t or "radome" in t or
        "cockpit" in t or "flight deck" in t or "flight compartment" in t or
        "windshield" in t or "windscreen" in t or
        "below captain's window" in t or "below first officer's window" in t):
        cats.add("nose")

    # ---------- FUSELAGE ----------
    if ("fuselage" in t or "airframe" in t or "aircraft body" in t or
        "airplane body" in t or "hull" in t or "pressure vessel" in t or
        "pressure body" in t or "cabin" in t or "belly" in t or
        "underside of fuselage" in t or "ventral" in t or
        "bulkhead" in t or "stringer" in t or "frame " in t or "skin" in t):
        cats.add("fuselage")

    # ---------- LANDING GEAR ----------
    if ("landing gear" in t or "landing-gear" in t or
        "undercarriage" in t or "gear " in t or t.startswith("gear") or
        "nose gear" in t or "nosewheel" in t or "nose wheel" in t or
        "main gear" in t or "main landing gear" in t or
        "mlg" in t or "nlg" in t or
        "strut" in t or "oleo" in t or "bogie" in t or
        "wheel well" in t or "wheel wells" in t or
        "wheel" in t or "wheels" in t or "tyre" in t or "tire" in t):
        cats.add("landing gear")

    return sorted(cats)


In [ ]:
new_final_df["Damage_Areas"] = new_final_df["damage_area"].apply(simplify_damage)

In [ ]:
comp_df = new_final_df[["Damage_Areas", "damage_area"]]

In [ ]:
new_final_df.columns

Index(['Event_No', 'Incident_Date', 'Year', 'Time', 'Incident',
       'Aircaft_Damage_Type', 'Aircraft_Phase', 'Occupants', 'Fatalities',
       'Ground_Casualties', 'Collision_Casualties', 'Incident_Location',
       'geopy_query', 'Latitude', 'Longitude', 'GeoStatus', 'Aircaft_Nature',
       'Aircaft_Registration', 'Aircaft_Model', 'Aircaft_Operator',
       'Engine_Model', 'Departure_Airport', 'Destination_Airport',
       'Departure_Airport_Code', 'Destination_Airport_Code', 'Departure_IATA',
       'Destination_IATA', 'Dep_Lat', 'Dep_Lon', 'Dest_Lat', 'Dest_Lon',
       'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating',
       'Narrative', 'Incident_Cause(es)', 'is_engine_cause', 'is_model_cause',
       'is_human_cause', 'is_weather_cause', 'is_wildlife_cause',
       'is_ground_collision', 'is_unknown_cause', 'damage_area',
       'Engine_Model_Standardized', 'Aircraft_Manufacturer',
       'Engine_Manufacturer', 'Damage_Areas'],
      dtype='object')

In [ ]:
new_final_df.to_csv("cleanest_df_with_NLP.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/oshani-jayawardane/aircraft-accident-project/main/cleanest_df_with_NLP.csv"
df = pd.read_csv(url)

df.head()

,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,Fatalities,Ground_Casualties,...,is_human_cause,is_weather_cause,is_wildlife_cause,is_ground_collision,is_unknown_cause,damage_area,Engine_Model_Standardized,Aircraft_Manufacturer,Engine_Manufacturer,Damage_Areas
0,1,2000-01-01,2000,13:00 LT,Accident,Substantial,Unknown,3.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,['right wing leading edge'],Pratt & Whitney JT15D-4,Cessna,Pratt & Whitney,['wing']
1,2,2000-01-03,2000,NaN,Unknown,Destroyed,Unknown,NaN,NaN,0.0,...,0.0,0.0,0.0,0.0,1.0,[],NaN,Beechcraft,NaN,[]
2,3,2000-01-04,2000,17:25 LT,Accident,Substantial,Unknown,3.0,0.0,0.0,...,1.0,1.0,0.0,1.0,0.0,['nose landing gear'],Pratt & Whitney PT6-A-42,Beechcraft,Pratt & Whitney,"['landing gear', 'nose']"
3,4,2000-01-05,2000,13:25,Accident,Destroyed,Approach,13.0,1.0,1.0,...,0.0,0.0,0.0,1.0,0.0,['farmland'],Pratt & Whitney Canada PT6A-34,Embraer,Pratt & Whitney,[]
4,5,2000-01-07,2000,NaN,Accident,Missing,Unknown,8.0,NaN,0.0,...,0.0,0.0,0.0,0.0,1.0,[],Ivchenko AI-24,Antonov,Ivchenko,[]


In [ ]:
from google.colab import files
import pandas as pd

# upload file
uploaded = files.upload()

# assuming only one file uploaded
file_name = list(uploaded.keys())[0]

# read into dataframe
other_df = pd.read_csv(file_name)

other_df.head()


Saving with_engine_aircraft_manufacturer (2).csv to with_engine_aircraft_manufacturer (2).csv


,Unnamed: 0.1,Unnamed: 0,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,...,Dest_Lon,Aircraft_Manufacture_Year,Total_Flight_Hours,Confidence_Rating,Narrative,Incident_Cause(es),Engine_Model_Standardized,Engine_Manufacturer_Robust,Aircraft_Manufacturer,Engine_Manufacturer
0,0,0,1,2000-01-01,2000,13:00 LT,Accident,Substantial,Unknown,3.0,...,56.24060,NaN,12159 hours,Accident investigation report completed and in...,The pilot-in-command (PIC) stated he was in cr...,NaN,Pratt & Whitney ratt & Whitney JT15D-4,Pratt & Whitney,Cessna,Pratt & Whitney
1,1,1,2,2000-01-03,2000,NaN,Unknown,Destroyed,Unknown,NaN,...,NaN,1978.0,NaN,Little or no information is available,Damaged beyond repair.,Info-Unavailable,NaN,NaN,Beechcraft,NaN
2,2,2,3,2000-01-04,2000,17:25 LT,Accident,Substantial,Unknown,3.0,...,92.49330,1986.0,3238 hours,Accident investigation report completed and in...,While performing the ILS runway 18 approach to...,NaN,Pratt & Whitney ratt & Whitney Pratt & Whitney...,Pratt & Whitney,Beechcraft,Pratt & Whitney
3,3,3,4,2000-01-05,2000,13:25,Accident,Destroyed,Approach,13.0,...,7.26317,1984.0,NaN,Information verified through data from acciden...,The Bandeirante aircraft was coming in to land...,Result - Loss of control,Pratt & Whitney ratt & Whitney Canada Pratt & ...,Pratt & Whitney,Embraer,Pratt & Whitney
4,4,4,5,2000-01-07,2000,NaN,Accident,Missing,Unknown,8.0,...,17.98970,1978.0,NaN,NaN,Disappeared near the border of the Angolan pro...,Unknown - Missing,Ivchenko AI-24,Ivchenko,Antonov,Ivchenko


In [ ]:
other_df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Event_No', 'Incident_Date', 'Year',
       'Time', 'Incident', 'Aircaft_Damage_Type', 'Aircraft_Phase',
       'Occupants', 'Fatalities', 'Ground_Casualties', 'Collision_Casualties',
       'Incident_Location', 'geopy_query', 'Latitude', 'Longitude',
       'GeoStatus', 'Aircaft_Nature', 'Aircaft_Registration', 'Aircaft_Model',
       'Aircaft_Operator', 'Engine_Model', 'Departure_Airport',
       'Destination_Airport', 'Departure_Airport_Code',
       'Destination_Airport_Code', 'Departure_IATA', 'Destination_IATA',
       'Dep_Lat', 'Dep_Lon', 'Dest_Lat', 'Dest_Lon',
       'Aircraft_Manufacture_Year', 'Total_Flight_Hours', 'Confidence_Rating',
       'Narrative', 'Incident_Cause(es)', 'Engine_Model_Standardized',
       'Engine_Manufacturer_Robust', 'Aircraft_Manufacturer',
       'Engine_Manufacturer'],
      dtype='object')

In [ ]:
other_df.head(20)

,Unnamed: 0.1,Unnamed: 0,Event_No,Incident_Date,Year,Time,Incident,Aircaft_Damage_Type,Aircraft_Phase,Occupants,...,Dest_Lon,Aircraft_Manufacture_Year,Total_Flight_Hours,Confidence_Rating,Narrative,Incident_Cause(es),Engine_Model_Standardized,Engine_Manufacturer_Robust,Aircraft_Manufacturer,Engine_Manufacturer
0,0,0,1,2000-01-01,2000,13:00 LT,Accident,Substantial,Unknown,3.0,...,56.24060,NaN,12159 hours,Accident investigation report completed and in...,The pilot-in-command (PIC) stated he was in cr...,NaN,Pratt & Whitney ratt & Whitney JT15D-4,Pratt & Whitney,Cessna,Pratt & Whitney
1,1,1,2,2000-01-03,2000,NaN,Unknown,Destroyed,Unknown,NaN,...,NaN,1978.0,NaN,Little or no information is available,Damaged beyond repair.,Info-Unavailable,NaN,NaN,Beechcraft,NaN
2,2,2,3,2000-01-04,2000,17:25 LT,Accident,Substantial,Unknown,3.0,...,92.49330,1986.0,3238 hours,Accident investigation report completed and in...,While performing the ILS runway 18 approach to...,NaN,Pratt & Whitney ratt & Whitney Pratt & Whitney...,Pratt & Whitney,Beechcraft,Pratt & Whitney
3,3,3,4,2000-01-05,2000,13:25,Accident,Destroyed,Approach,13.0,...,7.26317,1984.0,NaN,Information verified through data from acciden...,The Bandeirante aircraft was coming in to land...,Result - Loss of control,Pratt & Whitney ratt & Whitney Canada Pratt & ...,Pratt & Whitney,Embraer,Pratt & Whitney
4,4,4,5,2000-01-07,2000,NaN,Accident,Missing,Unknown,8.0,...,17.98970,1978.0,NaN,NaN,Disappeared near the border of the Angolan pro...,Unknown - Missing,Ivchenko AI-24,Ivchenko,Antonov,Ivchenko
5,5,5,6,2000-01-07,2000,15:44,Accident,Substantial,Landing,2.0,...,NaN,1970.0,NaN,Information verified through data from acciden...,"NWL100, a Beech King Air with 2 crew, was trai...",NaN,NaN,NaN,Beechcraft,NaN
6,6,6,7,2000-01-09,2000,18:00,Accident,Repairable,Approach,3.0,...,100.27700,1972.0,NaN,NaN,"On approach to Penang runway 22, the aircraft ...",Info-Unavailable,Pratt & Whitney ratt & Whitney JT9D-7A,Pratt & Whitney,Boeing,Pratt & Whitney
7,7,7,8,2000-01-10,2000,17:56,Accident,Destroyed,En Route,10.0,...,13.76720,1990.0,21674 hours,Accident investigation report completed and in...,"At 17:00 Saab 340 HB-AKK arrived at Zurich, Sw...",Result - Loss of control,General Electric neral Electric CT7-9B,General Electric,Saab,General Electric
8,8,8,9,2000-01-11,2000,22:51 LT,Accident,Unknown,En Route,44.0,...,NaN,1989.0,NaN,Accident investigation report completed and in...,One flight attendant sustained serious injurie...,NaN,Rolls-Royce e RB211-535E4,Rolls-Royce,Boeing,Rolls-Royce
9,9,9,10,2000-01-13,2000,14:38,Accident,Destroyed,Approach,41.0,...,19.57640,1990.0,7138 hours,Accident investigation report completed and in...,The Shorts 360 plane had been leased to Sirte ...,"Airplane - Engines, Airplane - Engines - All e...",Pratt & Whitney ratt & Whitney Canada Pratt & ...,Pratt & Whitney,Shorts,Pratt & Whitney


In [ ]:
cols = ["Engine_Model_Standardized", "Aircraft_Manufacturer", "Engine_Manufacturer"]

df[cols] = other_df[cols].values


In [ ]:
df.to_csv("updated_dataset.csv", index=False)
